# 02 — Tiền xử lý & đặc trưng

**Notebook gốc:** bước 2–3: kiểm tra chất lượng, lọc benign train, signed-log, loại feature hằng/hiếm/tương quan, chuẩn hóa. Các hàm nằm trong `src/preprocessing.py`. Notebook này là **kiểm tra độc lập**; notebook 03 sẽ fit lại pipeline khi chạy toàn bộ để bảo đảm tái lập.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Mở thư mục project trên VS Code, hoặc chạy notebook trong notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import config as cfg
print("Project:", PROJECT_ROOT)
print("Dataset:", cfg.DATA_DIR)


Project: D:\network-anomaly-hybrid-vscode\network-anomaly-hybrid-vscode
Dataset: D:\network-anomaly-hybrid-vscode\network-anomaly-hybrid-vscode\data\raw


In [2]:
import joblib
from pyspark.sql import functions as F
from src.spark_session import create_spark
from src.data_io import prepare_curated
from src.preprocessing import profile, valid_cond, fit_feature_spec
from src.utils import day_of

(cfg.OUT_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
(cfg.OUT_DIR / 'models').mkdir(parents=True, exist_ok=True)
spark = create_spark()
files = cfg.TRAIN_FILES + [cfg.VAL_FILE] + cfg.TEST_FILES
all_df = prepare_curated(spark, files)
feats = [c for c in all_df.columns if c not in cfg.EXCLUDE_FEATURES]
train_days = [day_of(p) for p in cfg.TRAIN_FILES]
ben_train_raw = all_df.filter(F.col('day').isin(train_days) & (F.col('y') == 0))
quality = profile(ben_train_raw, feats)
display(quality.sort_values('nonfinite', ascending=False).head(20))
quality.to_csv(cfg.OUT_DIR / 'metrics' / 'data_quality_train.csv', index=False)


,feature,nonfinite,neg,zero_frac,min,max,constant
0,flow_duration,0,5,0.000000,-9.190110e+11,1.200000e+08,False
50,down_up_ratio,0,0,0.445827,0.000000e+00,1.260000e+02,False
57,bwd_avg_bytes_bulk,0,0,1.000000,0.000000e+00,0.000000e+00,True
56,fwd_avg_bulk_rate,0,0,1.000000,0.000000e+00,0.000000e+00,True
55,fwd_avg_packets_bulk,0,0,1.000000,0.000000e+00,0.000000e+00,True
54,fwd_avg_bytes_bulk,0,0,1.000000,0.000000e+00,0.000000e+00,True
53,avg_bwd_segment_size,0,0,0.247779,0.000000e+00,1.459240e+03,False
52,avg_fwd_segment_size,0,0,0.165129,0.000000e+00,1.652931e+04,False
51,avg_packet_size,0,0,0.164732,0.000000e+00,3.328299e+03,False
49,ece_flag_count,0,0,0.916867,0.000000e+00,1.000000e+00,False


In [3]:
ben_train = ben_train_raw.filter(valid_cond(feats))
print('Benign ban đầu:', ben_train_raw.count(), '| hợp lệ:', ben_train.count())
cols, mean, std, feature_report = fit_feature_spec(ben_train, feats)
display(feature_report)
feature_report.to_csv(cfg.OUT_DIR / 'metrics' / 'feature_report.csv', index=False)
joblib.dump({'cols': cols, 'mean': mean, 'std': std, 'z_clip': cfg.Z_CLIP},
            cfg.OUT_DIR / 'models' / 'feature_spec.joblib')
print(f'Giữ {len(cols)}/{len(feats)} đặc trưng')


Benign ban đầu: 1268743 | hợp lệ: 1268738


,feature,decision,reason,std_slog,nonzero_frac,std_floored
0,flow_duration,keep,,4.544435,1.000000,False
1,total_fwd_packets,keep,,0.791535,1.000000,False
2,total_backward_packets,keep,,1.000812,0.849065,False
3,fwd_packets_length_total,keep,,2.383910,0.834874,False
4,bwd_packets_length_total,keep,,3.086852,0.752224,False
...,...,...,...,...,...,...
74,idle_max,drop,|rho|=1.000 với idle_mean,6.352727,0.172691,False
75,idle_min,drop,|rho|=1.000 với idle_mean,6.306909,0.172691,False
76,is_tcp,keep,,0.328733,0.658354,False
77,is_udp,drop,|rho|=0.991 với fwd_packet_length_min,0.322821,0.318082,False


Giữ 46/79 đặc trưng


**Quan trọng:** Không fit bước chọn/chuẩn hóa đặc trưng lên validation hoặc test. Cả notebook này và `run_pipeline.py` đều áp dụng cách đó.

In [4]:
spark.stop()